
# QML-SleepNet — Official x01–x35 Evaluation — FROZEN v2 (Pipeline-Valid UID Universe)

## Final evaluation contract

This notebook performs the one-time official Task-A evaluation of the already-frozen model.

### Why the evaluation universe is 17,248 rows

The frozen project protocol is based on **record-bounded complete 60-second ECG minute targets**.  
The coverage audit established:

- raw complete 60-second ECG minutes: **17,327**
- official annotation entries: **17,268**
- frozen Stage03 / final-prediction UIDs: **17,248**

The frozen UIDs are exactly the per-record intersection of:

1. a **fully observed raw 60-second ECG window**, and
2. an **available official minute annotation**.

Therefore:

- 20 official terminal annotations correspond to records whose raw ECG ends with only a **partial** final minute; those rows are **not fabricated/padded into the frozen Stage03/QML feature universe**.
- 79 complete raw minutes have no official annotation and therefore cannot be scored.
- the legitimate frozen, signal-backed, labelled evaluation universe is **17,248 rows**.

This notebook verifies that rule again from source artifacts before computing any metric.

### Non-negotiable rules

- CPU only is sufficient.
- No training.
- No inference regeneration.
- No architecture/feature changes.
- No threshold search or calibration fitting.
- Frozen threshold remains **0.415**.
- No silent intersection: the expected 17,248 UID universe is reconstructed explicitly and must match Stage03 and final predictions exactly.
- Official-test performance must not feed back into model development.


In [ ]:

# Cell 1 — environment and immutable artifact paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
RAW = ROOT / "data/raw/apnea_ecg"

FINAL_ROOT = ROOT / "outputs/GUIDE_EXACT_METRICMAX/05B_06_FINAL_GUIDE_CORRECTED_CONSOLIDATED_v1"
PRED_PATH = FINAL_ROOT / "final_operating_predictions.npz"
THRESH_PATH = FINAL_ROOT / "FINAL_OPERATING_THRESHOLD.json"
FINAL_MANIFEST = FINAL_ROOT / "FINAL_GUIDE_CORRECTED_MANIFEST.json"

STAGE03_INDEX = ROOT / "outputs/GUIDE_EXACT_METRICMAX/03_feature_bank_v1/guide_stage03_target_index.csv"

LABEL_JSON = RAW / "test_set_apnea_labels.json"
LABEL_TXT = RAW / "test-dataset-annos.txt"

OUT = ROOT / "outputs/GUIDE_EXACT_METRICMAX/FINAL_OFFICIAL_X_EVAL_FROZEN_v2_PIPELINE_VALID"
OUT.mkdir(parents=True, exist_ok=True)

RECORDS = [f"x{i:02d}" for i in range(1, 36)]
FS = 100
EPOCH_SECONDS = 60
EPOCH_SAMPLES = FS * EPOCH_SECONDS

for p in [PRED_PATH, THRESH_PATH, FINAL_MANIFEST, STAGE03_INDEX, LABEL_JSON, LABEL_TXT]:
    if not p.is_file():
        raise FileNotFoundError(p)
for rec in RECORDS:
    if not (RAW / f"{rec}.hea").is_file():
        raise FileNotFoundError(RAW / f"{rec}.hea")

print("CPU evaluation")
print("Training: NO")
print("Inference regeneration: NO")
print("Output:", OUT)


In [ ]:

# Cell 2 — helpers

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str))
    tmp.replace(path)

def atomic_csv(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def read_header(hea):
    first = Path(hea).read_text().splitlines()[0].strip()
    parts = first.split()
    if len(parts) < 4:
        raise RuntimeError(f"Malformed header: {hea}")
    fs = float(parts[2].split("/")[0])
    sig_len = int(parts[3])
    if abs(fs - FS) > 1e-9:
        raise RuntimeError(f"{hea}: fs={fs}, expected {FS}")
    return sig_len

def normalize_json_labels(obj):
    out = {}
    for rec, value in obj.items():
        if isinstance(value, str):
            seq = value.strip()
        elif isinstance(value, (list, tuple)):
            seq = "".join(map(str, value))
        else:
            raise TypeError(f"{rec}: unsupported JSON label type")
        if set(seq) - {"A", "N"}:
            raise RuntimeError(f"{rec}: unexpected label symbols")
        out[str(rec)] = seq
    return out

def parse_official_txt(path):
    out = {}
    current = None
    chunks = []
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if not line:
            continue
        if len(line) == 3 and line.startswith("x") and line[1:].isdigit():
            if current is not None:
                out[current] = "".join(chunks)
            current = line
            chunks = []
            continue
        if current is None:
            continue
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            seq = "".join(parts[1:]).strip()
            if set(seq) - {"A", "N"}:
                raise RuntimeError(f"Unexpected annotation line: {line!r}")
            chunks.append(seq)
    if current is not None:
        out[current] = "".join(chunks)
    return out

def metric_dict(y, prob, threshold):
    y = np.asarray(y, dtype=np.int8)
    prob = np.asarray(prob, dtype=np.float64)
    pred = (prob >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()

    return {
        "threshold": float(threshold),
        "n": int(len(y)),
        "positives": int(y.sum()),
        "negatives": int((1-y).sum()),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn / max(tn + fp, 1)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y, pred)),
        "auroc": float(roc_auc_score(y, prob)),
        "auprc": float(average_precision_score(y, prob)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [ ]:

# Cell 3 — verify frozen threshold/model/prediction artifacts BEFORE label use

threshold_meta = json.loads(THRESH_PATH.read_text())
final_manifest = json.loads(FINAL_MANIFEST.read_text())

if threshold_meta.get("selection_source") != "learning OOF only":
    raise RuntimeError("Threshold was not selected from learning OOF only.")
if threshold_meta.get("official_x_labels_used") is not False:
    raise RuntimeError("Threshold artifact label-seal invariant failed.")
if final_manifest.get("official_x_labels_used") is not False:
    raise RuntimeError("Final model manifest label-seal invariant failed.")

FROZEN_T = float(threshold_meta["frozen_threshold"])
if not np.isclose(FROZEN_T, 0.415, atol=1e-12, rtol=0):
    raise RuntimeError(f"Frozen threshold drift: {FROZEN_T}")

z = np.load(PRED_PATH, allow_pickle=False)
required = {"threshold", "test_uids", "test_probability", "test_prediction"}
missing = required - set(z.files)
if missing:
    raise RuntimeError(f"Prediction artifact missing keys: {sorted(missing)}")

artifact_t = float(np.asarray(z["threshold"]).item())
if not np.isclose(artifact_t, FROZEN_T, atol=1e-12, rtol=0):
    raise RuntimeError("Prediction artifact threshold mismatch.")

UID = np.asarray(z["test_uids"]).astype(str)
PROB = np.asarray(z["test_probability"], dtype=np.float64)
PRED = np.asarray(z["test_prediction"], dtype=np.int8)

if len(UID) != len(PROB) or len(UID) != len(PRED):
    raise RuntimeError("Frozen prediction vector length mismatch.")
if len(np.unique(UID)) != len(UID):
    raise RuntimeError("Duplicate frozen prediction UIDs.")
if not np.isfinite(PROB).all():
    raise RuntimeError("Non-finite frozen probabilities.")
if not np.array_equal(PRED, (PROB >= FROZEN_T).astype(np.int8)):
    raise RuntimeError("Stored hard predictions differ from frozen threshold application.")

print("Frozen threshold:", FROZEN_T)
print("Frozen prediction rows:", len(UID))
print("Prediction artifact audit: PASS")


In [ ]:

# Cell 4 — open official annotation sources once and verify they agree

labels_json = normalize_json_labels(json.loads(LABEL_JSON.read_text()))
labels_txt = parse_official_txt(LABEL_TXT)

if sorted(labels_json) != RECORDS or sorted(labels_txt) != RECORDS:
    raise RuntimeError("Official label record universe is not exactly x01-x35.")

for rec in RECORDS:
    if labels_json[rec] != labels_txt[rec]:
        raise RuntimeError(f"Official annotation sources disagree for {rec}")

atomic_json(
    OUT / "OFFICIAL_LABEL_OPEN_MARKER.json",
    {
        "opened_at_utc": datetime.now(timezone.utc).isoformat(),
        "purpose": "one-time final frozen evaluation",
        "label_json_sha256": sha256_file(LABEL_JSON),
        "label_txt_sha256": sha256_file(LABEL_TXT),
        "model_frozen_before_label_open": True,
        "threshold_frozen_before_label_open": FROZEN_T,
        "model_development_reopened": False,
    }
)

print("Official annotation sources: EXACT MATCH")
print("Total annotation entries:", sum(len(labels_json[r]) for r in RECORDS))


In [ ]:

# Cell 5 — reconstruct the legitimate frozen evaluation UID universe from raw duration + annotation availability

expected_rows = []
excluded_partial_label_rows = []
complete_raw_unlabelled_rows = []
coverage_rows = []

for rec in RECORDS:
    sig_len = read_header(RAW / f"{rec}.hea")
    full_raw = sig_len // EPOCH_SAMPLES
    tail_samples = sig_len % EPOCH_SAMPLES
    n_labels = len(labels_json[rec])

    # A scorable minute must have BOTH:
    #   (a) a complete observed raw 60-second ECG window
    #   (b) an official annotation
    n_expected = min(full_raw, n_labels)

    for ep in range(n_expected):
        expected_rows.append({
            "uid": f"{rec}:{ep}",
            "record_name": rec,
            "epoch_idx": ep,
        })

    # Official labels beyond complete observed raw minutes:
    for ep in range(full_raw, n_labels):
        excluded_partial_label_rows.append({
            "uid": f"{rec}:{ep}",
            "record_name": rec,
            "epoch_idx": ep,
            "raw_tail_samples": tail_samples,
            "raw_tail_seconds": tail_samples / FS,
            "reason": "official annotation exists but no complete observed raw 60-second ECG window",
        })

    # Complete raw minutes without an official annotation:
    for ep in range(n_labels, full_raw):
        complete_raw_unlabelled_rows.append({
            "uid": f"{rec}:{ep}",
            "record_name": rec,
            "epoch_idx": ep,
            "reason": "complete raw 60-second ECG exists but no official annotation",
        })

    coverage_rows.append({
        "record_name": rec,
        "raw_complete_60s_minutes": int(full_raw),
        "official_annotation_count": int(n_labels),
        "expected_scorable_minutes": int(n_expected),
        "raw_tail_samples": int(tail_samples),
        "raw_tail_seconds": float(tail_samples / FS),
    })

expected_df = pd.DataFrame(expected_rows)
partial_df = pd.DataFrame(excluded_partial_label_rows)
unlabelled_df = pd.DataFrame(complete_raw_unlabelled_rows)
coverage_df = pd.DataFrame(coverage_rows)

EXPECTED_UID = expected_df["uid"].astype(str).to_numpy()

print("Raw complete minute total:", int(coverage_df["raw_complete_60s_minutes"].sum()))
print("Official annotation total:", int(coverage_df["official_annotation_count"].sum()))
print("Expected scorable intersection:", len(EXPECTED_UID))
print("Labelled-but-partial terminal rows:", len(partial_df))
print("Complete-raw-but-unlabelled rows:", len(unlabelled_df))

if len(EXPECTED_UID) != 17248:
    raise RuntimeError(f"Expected scorable universe drift: {len(EXPECTED_UID)} != 17248")

atomic_csv(OUT / "official_x_coverage_contract.csv", coverage_df)
atomic_csv(OUT / "excluded_labelled_partial_terminal_minutes.csv", partial_df)
atomic_csv(OUT / "complete_raw_but_unlabelled_minutes.csv", unlabelled_df)


In [ ]:

# Cell 6 — exact frozen UID identity gate: reconstructed universe == Stage03 == final predictions

s3 = pd.read_csv(STAGE03_INDEX)
if not {"uid", "record_name", "epoch_idx"}.issubset(s3.columns):
    raise RuntimeError("Stage03 index missing UID/accounting columns.")

s3x = s3[s3["record_name"].astype(str).isin(RECORDS)].copy()
S3_UID = s3x["uid"].astype(str).to_numpy()

if len(S3_UID) != 17248:
    raise RuntimeError(f"Stage03 official-x row count drift: {len(S3_UID)}")
if len(UID) != 17248:
    raise RuntimeError(f"Final frozen prediction count drift: {len(UID)}")

if not np.array_equal(S3_UID, EXPECTED_UID):
    raise RuntimeError(
        "Stage03 official-x UID universe/order does not equal reconstructed "
        "complete-raw + labelled evaluation universe."
    )

if not np.array_equal(UID, EXPECTED_UID):
    raise RuntimeError(
        "Final frozen prediction UID universe/order does not equal reconstructed "
        "complete-raw + labelled evaluation universe."
    )

if not np.array_equal(UID, S3_UID):
    raise RuntimeError("Stage03 and final-prediction UID universes differ.")

coverage_manifest = {
    "status": "PASS",
    "rule": "scorable iff complete observed raw 60-second ECG window AND official annotation both exist",
    "raw_complete_minutes_total": int(coverage_df["raw_complete_60s_minutes"].sum()),
    "official_annotation_entries_total": int(coverage_df["official_annotation_count"].sum()),
    "scorable_frozen_uid_total": int(len(EXPECTED_UID)),
    "labelled_but_partial_terminal_excluded": int(len(partial_df)),
    "complete_raw_but_unlabelled_excluded": int(len(unlabelled_df)),
    "stage03_uid_total": int(len(S3_UID)),
    "final_prediction_uid_total": int(len(UID)),
    "exact_uid_order_match": True,
    "row_fabrication": False,
    "partial_tail_padding_added_for_evaluation": False,
}
atomic_json(OUT / "OFFICIAL_X_EVALUATION_UNIVERSE_MANIFEST.json", coverage_manifest)

print(json.dumps(coverage_manifest, indent=2))
print("\nEXACT EVALUATION-UNIVERSE IDENTITY: PASS")


In [ ]:

# Cell 7 — map official labels onto the already-frozen 17,248 UIDs and compute primary metrics

label_by_uid = {}
for rec in RECORDS:
    seq = labels_json[rec]
    for ep, symbol in enumerate(seq):
        label_by_uid[f"{rec}:{ep}"] = 1 if symbol == "A" else 0

# Every frozen UID was already proven to have an annotation.
Y = np.asarray([label_by_uid[u] for u in UID], dtype=np.int8)

METRICS = metric_dict(Y, PROB, FROZEN_T)
atomic_json(OUT / "OFFICIAL_X_PRIMARY_METRICS.json", METRICS)

pred_df = pd.DataFrame({
    "uid": UID,
    "record_name": [u.rsplit(":",1)[0] for u in UID],
    "epoch_idx": [int(u.rsplit(":",1)[1]) for u in UID],
    "y_true": Y,
    "probability": PROB,
    "prediction": PRED,
})
atomic_csv(OUT / "OFFICIAL_X_PRIMARY_PREDICTIONS.csv", pred_df)

print("=" * 90)
print("FINAL OFFICIAL-X TASK-A METRICS — FROZEN 17,248-UID PIPELINE-VALID UNIVERSE")
print("=" * 90)
print(json.dumps(METRICS, indent=2))


In [ ]:

# Cell 8 — plots

cm = confusion_matrix(Y, PRED, labels=[0,1])

fig, ax = plt.subplots(figsize=(5.5,5))
ax.imshow(cm)
ax.set_title("Official x01-x35 · Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks([0,1], ["Normal","Apnea"])
ax.set_yticks([0,1], ["Normal","Apnea"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center")
fig.tight_layout()
fig.savefig(OUT / "official_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

fpr, tpr, _ = roc_curve(Y, PROB)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, label=f"AUROC={METRICS['auroc']:.4f}")
ax.plot([0,1],[0,1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Official ROC")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "official_roc.png", dpi=180, bbox_inches="tight")
plt.show()

prec, rec, _ = precision_recall_curve(Y, PROB)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(rec, prec, label=f"AUPRC={METRICS['auprc']:.4f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Official Precision-Recall")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "official_pr.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:

# Cell 9 — per-record descriptive metrics

rows = []

for rec, q in pred_df.groupby("record_name", sort=True):
    y = q["y_true"].to_numpy(np.int8)
    p = q["probability"].to_numpy(float)
    pred = q["prediction"].to_numpy(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()

    rows.append({
        "record_name": rec,
        "n": int(len(q)),
        "positive_minutes": int(y.sum()),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": (
            float(balanced_accuracy_score(y, pred))
            if len(np.unique(y)) > 1 else np.nan
        ),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn / max(tn+fp, 1)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "mcc": (
            float(matthews_corrcoef(y, pred))
            if len(np.unique(y)) > 1 and len(np.unique(pred)) > 1 else np.nan
        ),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "auprc": float(average_precision_score(y, p)) if y.sum() > 0 else np.nan,
    })

record_metrics = pd.DataFrame(rows)
atomic_csv(OUT / "OFFICIAL_X_PER_RECORD_METRICS.csv", record_metrics)
display(record_metrics)

record_summary = {
    "record_macro_accuracy": float(record_metrics["accuracy"].mean()),
    "record_macro_balanced_accuracy_valid": float(record_metrics["balanced_accuracy"].mean(skipna=True)),
    "record_macro_f1": float(record_metrics["f1"].mean()),
    "record_macro_auroc_valid": float(record_metrics["auroc"].mean(skipna=True)),
    "record_macro_auprc_valid": float(record_metrics["auprc"].mean(skipna=True)),
}
atomic_json(OUT / "OFFICIAL_X_RECORD_MACRO_SUMMARY.json", record_summary)
print(json.dumps(record_summary, indent=2))


In [ ]:

# Cell 10 — permanent final-evaluation manifest

manifest = {
    "status": "OFFICIAL_X_FROZEN_EVALUATION_COMPLETE",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "task": "Task A binary Apnea vs Normal",
    "evaluation_universe_definition": (
        "record-bounded minute UID with both a complete observed raw 60-second ECG window "
        "and an available official x01-x35 annotation"
    ),
    "n_evaluated": int(len(Y)),
    "n_official_annotation_entries_total": int(coverage_df["official_annotation_count"].sum()),
    "n_raw_complete_60s_minutes_total": int(coverage_df["raw_complete_60s_minutes"].sum()),
    "n_labelled_partial_terminal_excluded": int(len(partial_df)),
    "n_complete_raw_unlabelled_excluded": int(len(unlabelled_df)),
    "threshold": FROZEN_T,
    "threshold_source": threshold_meta.get("selection_source"),
    "prediction_artifact": str(PRED_PATH),
    "prediction_sha256": sha256_file(PRED_PATH),
    "threshold_artifact": str(THRESH_PATH),
    "threshold_sha256": sha256_file(THRESH_PATH),
    "stage03_index": str(STAGE03_INDEX),
    "stage03_index_sha256": sha256_file(STAGE03_INDEX),
    "label_json_sha256": sha256_file(LABEL_JSON),
    "label_txt_sha256": sha256_file(LABEL_TXT),
    "uid_universe_exactly_reconstructed": True,
    "model_changed_after_label_open": False,
    "model_retrained_after_label_open": False,
    "threshold_changed_after_label_open": False,
    "test_tuning_performed": False,
    "partial_tail_predictions_fabricated": False,
    "primary_metrics": METRICS,
}

atomic_json(OUT / "OFFICIAL_X_FINAL_EVALUATION_MANIFEST.json", manifest)

print("=" * 96)
print("OFFICIAL X01-X35 FROZEN EVALUATION COMPLETE")
print("=" * 96)
print(json.dumps(METRICS, indent=2))
print("\nSaved to:", OUT)



## What to send back

After **Run all**, send:

1. the final JSON printed by **Cell 7** (`METRICS`), and
2. the final JSON printed by **Cell 9** (`record_summary`).

Do not retrain or tune anything after seeing these results.
